# Phase 04G: Teacher B → S2 KD Compatibility Entry

Phase 04F đã hoàn tất Teacher A → S1 direct Logit-KD. Cặp tiếp theo:

- Teacher B: BLIP Base, native BERT/WordPiece output space;
- Student S2: MobileViT-S + shared word-level decoder;
- matched seed phát triển: 42;
- Held-out Test luôn `SEALED`.

## Vấn đề cần giải quyết

Teacher B có output vocabulary 30.524 token, Student S2 có 10.080 token.
Phase 03H đã khóa:

- direct Logit-KD: **không được phép**;
- Sequence KD: **được phép**;
- Feature KD: chỉ được phép sau projection/tensor contract riêng.

Vì vậy Phase 04G không huấn luyện. Nó khóa phương pháp reproduction hợp lệ:
Teacher sinh pseudo caption, pseudo caption được tokenize lại bằng tokenizer
của Student, rồi Student tối ưu CE trên human caption và teacher sequence.

## Kaggle inputs

1. Phase 04F Teacher A → S1 aggregate.
2. Phase 03H Teacher B three-seed aggregate.
3. Phase 03G Teacher B seed-42 output.
4. Phase 02A shared Student bundle.
5. Phase 02C S2 seed-42 baseline output.
6. Phase 01 formal output.

CPU là đủ; Internet Off. Không mở Test Parquet.


## Cell 1 — Authorization và formal input discovery

Khóa đúng Teacher B, Student S2, seed 42 và các leakage boundaries.


In [1]:
import csv
import hashlib
import json
import math
import os
import time
from datetime import datetime, timezone
from pathlib import Path


CELL_STARTED_AT = time.perf_counter()
INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path(
    "/kaggle/working//"
    "phase4_teacher_b_s2_kd_compatibility_entry"
)
AUDIT_DIR = OUTPUT_ROOT / "audits"
CONFIG_DIR = OUTPUT_ROOT / "configs"
MANIFEST_DIR = OUTPUT_ROOT / "manifests"
for directory in [
    OUTPUT_ROOT, AUDIT_DIR, CONFIG_DIR, MANIFEST_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

EXPERIMENT_ID = "P4B_B_S2_SEQKD"
SEED = 42


def now_utc():
    return datetime.now(timezone.utc).isoformat()


def read_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)


def write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)


def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def exactly_one(name):
    matches = sorted(INPUT_ROOT.rglob(name))
    assert len(matches) == 1, (
        f"Cần đúng một {name}; tìm thấy {len(matches)}: {matches}"
    )
    return matches[0].resolve()


def resolve_inside(root, relative_path):
    root = Path(root).resolve()
    target = (root / relative_path).resolve()
    assert target.is_relative_to(root), (root, target)
    return target


PHASE4F_COMPLETE_PATH = exactly_one(
    "PHASE4_TEACHER_A_S1_LOGIT_KD_"
    "THREE_SEED_AGGREGATE_COMPLETE.json"
)
ENTRY_AUTHORIZATION_PATH = exactly_one(
    "PHASE4_TEACHER_B_S2_LOGIT_KD_ENTRY_AUTHORIZED.json"
)
TEACHER_B_AGGREGATE_PATH = exactly_one(
    "PHASE3_TEACHER_B_THREE_SEED_AGGREGATE_COMPLETE.json"
)
TEACHER_B_SEED42_PATH = exactly_one(
    "TEACHER_B_SEED42_FULL_RUN_COMPLETE.json"
)
PHASE2_READY_PATH = exactly_one(
    "PHASE2_SHARED_TRAINING_BUNDLE_READY.json"
)
S2_BASELINE_PATH = exactly_one(
    "S2_SEED42_FULL_RUN_COMPLETE.json"
)
PHASE1_READY_PATH = exactly_one(
    "PHASE1_FORMAL_DELIVERABLES_READY.json"
)

phase4f = read_json(PHASE4F_COMPLETE_PATH)
entry_authorization = read_json(ENTRY_AUTHORIZATION_PATH)
teacher_aggregate = read_json(TEACHER_B_AGGREGATE_PATH)
teacher_seed42 = read_json(TEACHER_B_SEED42_PATH)
phase2_ready = read_json(PHASE2_READY_PATH)
s2_baseline = read_json(S2_BASELINE_PATH)
phase1_ready = read_json(PHASE1_READY_PATH)

assert phase4f["status"] == "complete"
assert phase4f["reproduction_status"] == "passed"
assert phase4f["test_set_status"] == "sealed"
assert phase4f["test_set_used"] is False
assert entry_authorization["status"] == "authorized"
assert entry_authorization["source_reproduction_status"] == "passed"
assert entry_authorization["matched_student"] == "S2 — MobileViT-S"
assert entry_authorization["method_changes_allowed"] is False
assert teacher_aggregate["status"] == "complete"
assert teacher_aggregate["teacher_id"] == "B"
assert teacher_aggregate["all_qualified_for_kd"] is True
assert teacher_seed42["status"] == "complete"
assert teacher_seed42["teacher_id"] == "B"
assert int(teacher_seed42["seed"]) == SEED
assert teacher_seed42["qualified_for_kd"] is True
assert phase2_ready["status"] == "ready"
assert "S2" in phase2_ready["student_ids"]
assert s2_baseline["status"] == "complete"
assert s2_baseline["student_id"] == "S2"
assert int(s2_baseline["seed"]) == SEED
assert phase1_ready["status"] == "ready"

for marker in [
    phase4f, entry_authorization, teacher_aggregate,
    teacher_seed42, s2_baseline,
]:
    assert marker["test_set_status"] == "sealed"
    assert marker["test_set_used"] is False
assert phase1_ready["test_set_status"] == "sealed"
assert phase1_ready["test_used"] is False

write_json(
    AUDIT_DIR / "phase4g_cell01_inputs.json",
    {
        "audit_version": "phase4g_cell01_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "experiment_id": EXPERIMENT_ID,
        "teacher_id": "B",
        "student_id": "S2",
        "development_seed": SEED,
        "phase4f_reproduction": "passed",
        "teacher_b_three_seed": "verified",
        "teacher_b_seed42": "qualified_for_kd",
        "s2_seed42_baseline": "verified",
        "dataset_read": False,
        "caption_read": False,
        "checkpoint_payload_read": False,
        "test_parquet_opened": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 04G CELL 1 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Teacher / Student         : B / S2")
print("Development seed          : 42")
print("Phase 04F reproduction    : PASSED")
print("Teacher B aggregate       : VERIFIED")
print("S2 baseline               : VERIFIED")
print("Dataset/caption read      : False / False")
print("Checkpoint payload read   : False")
print("Test Set status           : SEALED")
print("Next                      : verify formal artifacts")


PHASE 04G CELL 1 STATUS
Status                    : PASSED
Teacher / Student         : B / S2
Development seed          : 42
Phase 04F reproduction    : PASSED
Teacher B aggregate       : VERIFIED
S2 baseline               : VERIFIED
Dataset/caption read      : False / False
Checkpoint payload read   : False
Test Set status           : SEALED
Next                      : verify formal artifacts


## Cell 2 — Manifest, checkpoint và vocabulary artifacts

Xác minh Phase 04F, Phase 03H, Phase 02A; khóa Teacher seed-42 checkpoint,
Student S2 initialization và vocabulary Phase 1. Chưa deserialize checkpoint.


In [2]:
CELL_STARTED_AT = time.perf_counter()


def verify_manifest(root, csv_path, sums_path):
    rows = list(csv.DictReader(
        csv_path.open("r", encoding="utf-8", newline="")
    ))
    expected_lines = []
    for row in rows:
        artifact = resolve_inside(root, row["relative_path"])
        assert artifact.stat().st_size == int(row["size_bytes"])
        assert sha256_file(artifact) == row["sha256"]
        expected_lines.append(
            f'{row["sha256"]}  {row["relative_path"]}'
        )
    actual_lines = [
        line.strip()
        for line in sums_path.read_text(
            encoding="utf-8"
        ).splitlines()
        if line.strip()
    ]
    assert actual_lines == expected_lines
    return rows


PHASE4F_ROOT = PHASE4F_COMPLETE_PATH.parent
phase4f_manifest = resolve_inside(
    PHASE4F_ROOT,
    phase4f["manifest"]["csv"]["relative_path"],
)
phase4f_sums = resolve_inside(
    PHASE4F_ROOT,
    phase4f["manifest"]["sha256sums"]["relative_path"],
)
assert sha256_file(phase4f_manifest) == (
    phase4f["manifest"]["csv"]["sha256"]
)
phase4f_rows = verify_manifest(
    PHASE4F_ROOT, phase4f_manifest, phase4f_sums
)
assert len(phase4f_rows) == 13

TEACHER_AGGREGATE_ROOT = TEACHER_B_AGGREGATE_PATH.parent
teacher_manifest = resolve_inside(
    TEACHER_AGGREGATE_ROOT,
    teacher_aggregate["manifest"]["csv"]["relative_path"],
)
teacher_sums = resolve_inside(
    TEACHER_AGGREGATE_ROOT,
    teacher_aggregate["manifest"][
        "sha256sums"
    ]["relative_path"],
)
assert sha256_file(teacher_manifest) == (
    teacher_aggregate["manifest"]["csv"]["sha256"]
)
teacher_rows = verify_manifest(
    TEACHER_AGGREGATE_ROOT,
    teacher_manifest,
    teacher_sums,
)
assert len(teacher_rows) == 13

PHASE2_ROOT = PHASE2_READY_PATH.parent
phase2_manifest = resolve_inside(
    PHASE2_ROOT,
    phase2_ready["manifests"]["csv"]["relative_path"],
)
phase2_sums = resolve_inside(
    PHASE2_ROOT,
    phase2_ready["manifests"]["sha256sums"]["relative_path"],
)
assert sha256_file(phase2_manifest) == (
    phase2_ready["manifests"]["csv"]["sha256"]
)
phase2_rows = verify_manifest(
    PHASE2_ROOT, phase2_manifest, phase2_sums
)
assert len(phase2_rows) == 38
phase2_by_path = {
    row["relative_path"]: row for row in phase2_rows
}

S2_INITIAL_CHECKPOINT = resolve_inside(
    PHASE2_ROOT,
    phase2_ready["required_artifacts"]["s2_encoder_checkpoint"],
)
SHARED_DECODER_CHECKPOINT = resolve_inside(
    PHASE2_ROOT,
    phase2_ready["required_artifacts"][
        "shared_decoder_checkpoint"
    ],
)
for checkpoint in [
    S2_INITIAL_CHECKPOINT, SHARED_DECODER_CHECKPOINT
]:
    relative = checkpoint.relative_to(PHASE2_ROOT).as_posix()
    assert relative in phase2_by_path
    assert sha256_file(checkpoint) == (
        phase2_by_path[relative]["sha256"]
    )

registry_path = resolve_inside(
    TEACHER_AGGREGATE_ROOT,
    teacher_aggregate["checkpoint_registry"]["relative_path"],
)
assert sha256_file(registry_path) == (
    teacher_aggregate["checkpoint_registry"]["sha256"]
)
teacher_registry = read_json(registry_path)
assert teacher_registry["status"] == "locked"
assert teacher_registry["entry_count"] == 3
seed42_registry = [
    item for item in teacher_registry["entries"]
    if int(item["seed"]) == SEED
][0]

TEACHER_B_SEED42_ROOT = TEACHER_B_SEED42_PATH.parent
TEACHER_B_BEST_CHECKPOINT = (
    TEACHER_B_SEED42_ROOT
    / teacher_seed42["best_checkpoint"]["relative_path"]
)
assert TEACHER_B_BEST_CHECKPOINT.is_file()
assert sha256_file(TEACHER_B_BEST_CHECKPOINT) == (
    teacher_seed42["best_checkpoint"]["sha256"]
)
assert sha256_file(TEACHER_B_BEST_CHECKPOINT) == (
    seed42_registry["best_checkpoint"]["sha256"]
)

VOCABULARY_PATH = exactly_one(
    "word_vocabulary_minfreq5.json"
)
vocabulary = read_json(VOCABULARY_PATH)
assert vocabulary["vocabulary_size"] == 10080
assert vocabulary["minimum_frequency"] == 5
assert vocabulary["special_token_ids"] == {
    "<pad>": 0, "<bos>": 1, "<eos>": 2, "<unk>": 3
}

write_json(
    AUDIT_DIR / "phase4g_cell02_artifacts.json",
    {
        "audit_version": "phase4g_cell02_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "phase4f_manifest_rows": len(phase4f_rows),
        "teacher_aggregate_manifest_rows": len(teacher_rows),
        "phase2_manifest_rows": len(phase2_rows),
        "teacher_checkpoint_sha256": sha256_file(
            TEACHER_B_BEST_CHECKPOINT
        ),
        "student_encoder_sha256": sha256_file(
            S2_INITIAL_CHECKPOINT
        ),
        "student_decoder_sha256": sha256_file(
            SHARED_DECODER_CHECKPOINT
        ),
        "student_vocabulary_size": vocabulary["vocabulary_size"],
        "checkpoint_payload_read": False,
        "test_parquet_opened": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 04G CELL 2 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Phase 04F manifest        : 13 / 13")
print("Teacher aggregate manifest: 13 / 13")
print("Phase 02A manifest        : 38 / 38")
print("Teacher B best checkpoint : VERIFIED")
print("S2 initial checkpoints    : 2 / 2 VERIFIED")
print("Student vocabulary        : 10080")
print("Checkpoint payload read   : False")
print("Test Set status           : SEALED")
print("Next                      : output-space compatibility")


PHASE 04G CELL 2 STATUS
Status                    : PASSED
Phase 04F manifest        : 13 / 13
Teacher aggregate manifest: 13 / 13
Phase 02A manifest        : 38 / 38
Teacher B best checkpoint : VERIFIED
S2 initial checkpoints    : 2 / 2 VERIFIED
Student vocabulary        : 10080
Checkpoint payload read   : False
Test Set status           : SEALED
Next                      : output-space compatibility


## Cell 3 — Output-space compatibility decision

Đọc compatibility contract của Phase 03H và chứng minh direct KL không hợp lệ
do hai output distributions không cùng index semantics.


In [3]:
CELL_STARTED_AT = time.perf_counter()
entries = teacher_registry["entries"]
assert [int(item["seed"]) for item in entries] == [42, 123, 2026]
for item in entries:
    compatibility = item["kd_compatibility"]
    assert compatibility["direct_logit_kd"]["allowed"] is False
    assert compatibility["sequence_kd"]["allowed"] is True
    assert compatibility["feature_kd"]["allowed"] is True
    assert item["test_set_status"] == "sealed"
    assert item["test_set_used"] is False

teacher_model_audits = sorted(
    TEACHER_B_SEED42_ROOT.rglob(
        "phase3g_cell03_model_data.json"
    )
)
assert len(teacher_model_audits) == 1
teacher_model_audit = read_json(teacher_model_audits[0])
assert teacher_model_audit["status"] == "passed"
TEACHER_VOCABULARY_SIZE = int(
    teacher_model_audit["tokenizer_size"]
)
STUDENT_VOCABULARY_SIZE = int(
    vocabulary["vocabulary_size"]
)
assert TEACHER_VOCABULARY_SIZE == 30524
assert STUDENT_VOCABULARY_SIZE == 10080
assert TEACHER_VOCABULARY_SIZE != STUDENT_VOCABULARY_SIZE

teacher_cider = float(
    teacher_seed42["best_result"]["CIDEr-D"]
)
student_cider = float(
    s2_baseline["best_validation_metrics"]["CIDEr-D"]
)
quality_margin = teacher_cider - student_cider
assert quality_margin > 0.001

COMPATIBILITY_DECISION = {
    "schema_version": "phase4g_kd_compatibility_v1",
    "created_at_utc": now_utc(),
    "status": "locked",
    "experiment_id": EXPERIMENT_ID,
    "teacher": {
        "id": "B",
        "model": teacher_seed42["model"],
        "seed": 42,
        "native_tokenizer": "BERT WordPiece",
        "vocabulary_size": TEACHER_VOCABULARY_SIZE,
        "validation_cider_d": teacher_cider,
    },
    "student": {
        "id": "S2",
        "model": "MobileViT-S + shared Transformer decoder",
        "seed": 42,
        "native_tokenizer": "Phase 1 deterministic word-level",
        "vocabulary_size": STUDENT_VOCABULARY_SIZE,
        "validation_cider_d": student_cider,
    },
    "quality_margin_cider_d": quality_margin,
    "methods": {
        "direct_logit_kd": {
            "status": "prohibited",
            "reason": (
                "Teacher and Student logits have different dimensions "
                "and different token-index semantics; KL is undefined "
                "without an explicit vocabulary bridge."
            ),
        },
        "sequence_kd": {
            "status": "selected_for_phase4_reproduction",
            "reason": (
                "Teacher captions are ordinary text and can be "
                "deterministically re-tokenized by the Student."
            ),
        },
        "feature_kd": {
            "status": "deferred_to_phase5_improvement",
            "reason": (
                "Requires a learned projection and adds an architectural "
                "component beyond reproduction."
            ),
        },
    },
    "test_set_status": "sealed",
    "test_set_used": False,
}
COMPATIBILITY_PATH = CONFIG_DIR / (
    "teacher_b_s2_kd_compatibility_contract.json"
)
write_json(COMPATIBILITY_PATH, COMPATIBILITY_DECISION)

write_json(
    AUDIT_DIR / "phase4g_cell03_compatibility.json",
    {
        "audit_version": "phase4g_cell03_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "teacher_vocabulary_size": TEACHER_VOCABULARY_SIZE,
        "student_vocabulary_size": STUDENT_VOCABULARY_SIZE,
        "direct_logit_kd": "prohibited",
        "sequence_kd": "selected",
        "feature_kd": "deferred",
        "teacher_student_cider_margin": quality_margin,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 04G OUTPUT-SPACE COMPATIBILITY")
print("=" * 72)
print("Status                    : PASSED")
print("Teacher vocabulary        : 30524")
print("Student vocabulary        : 10080")
print("Direct Logit-KD           : PROHIBITED")
print("Sequence KD               : SELECTED")
print("Feature KD                : DEFERRED TO PHASE 5")
print("Teacher CIDEr-D           :", teacher_cider)
print("S2 baseline CIDEr-D       :", student_cider)
print("Quality margin            :", quality_margin)
print("Test Set status           : SEALED")
print("Next                      : lock sequence-KD preflight policy")


PHASE 04G OUTPUT-SPACE COMPATIBILITY
Status                    : PASSED
Teacher vocabulary        : 30524
Student vocabulary        : 10080
Direct Logit-KD           : PROHIBITED
Sequence KD               : SELECTED
Feature KD                : DEFERRED TO PHASE 5
Teacher CIDEr-D           : 1.3301650423566496
S2 baseline CIDEr-D       : 0.9973554396007744
Quality margin            : 0.3328096027558751
Test Set status           : SEALED
Next                      : lock sequence-KD preflight policy


## Cell 4 — Sequence-KD reproduction policy

Khóa cách tạo pseudo caption, tokenizer boundary, loss và các gate cho Phase
04H. Temperature không áp dụng vì không so sánh logits.


In [4]:
CELL_STARTED_AT = time.perf_counter()
SEQUENCE_KD_POLICY = {
    "schema_version": "phase4g_sequence_kd_policy_v1",
    "created_at_utc": now_utc(),
    "status": "locked",
    "experiment_id": EXPERIMENT_ID,
    "teacher": {
        "id": "B",
        "model": "Salesforce/blip-image-captioning-base",
        "revision": teacher_seed42["revision"],
        "seed": 42,
        "checkpoint_sha256": sha256_file(
            TEACHER_B_BEST_CHECKPOINT
        ),
        "mode": "eval_frozen_no_grad",
        "generation": {
            "beam_size": 3,
            "do_sample": False,
            "length_penalty": 0.7,
            "minimum_new_tokens": 2,
            "maximum_new_tokens": 31,
        },
    },
    "student": {
        "id": "S2",
        "model": "MobileViT-S + shared Transformer decoder",
        "initial_encoder_sha256": sha256_file(
            S2_INITIAL_CHECKPOINT
        ),
        "initial_decoder_sha256": sha256_file(
            SHARED_DECODER_CHECKPOINT
        ),
        "vocabulary_size": 10080,
        "maximum_lexical_tokens": 30,
        "maximum_decoder_steps": 31,
    },
    "supervision": {
        "human_caption": (
            "one cyclic Train+Restval caption per image per epoch"
        ),
        "teacher_caption": (
            "one deterministic pseudo caption per Train+Restval image"
        ),
        "student_retokenization": (
            "Phase 1 deterministic word-level tokenizer"
        ),
        "loss": (
            "L = (1-alpha) * CE_human + alpha * CE_teacher_sequence"
        ),
        "alpha": 0.5,
        "temperature": None,
        "direct_teacher_logits_used": False,
    },
    "data_boundary": {
        "pseudo_caption_split": "Train+Restval only",
        "validation_pseudo_caption_used_for_training": False,
        "held_out_test_opened": False,
        "test_set_status": "sealed",
    },
    "phase04h_preflight": {
        "fixed_sample_images": 64,
        "required_checks": [
            "teacher deterministic generation repeatability",
            "pseudo-caption nonempty and finite tokenization",
            "student OOV and truncation diagnostics",
            "combined-loss forward/backward",
            "teacher frozen and gradient-free",
            "student gradients finite and nonzero",
            "discarded optimizer-step pilot",
            "atomic checkpoint/resume exactness",
        ],
        "training_started": False,
    },
    "deferred_method": {
        "feature_kd": "Phase 5 improved KD only",
    },
}
SEQUENCE_POLICY_PATH = CONFIG_DIR / (
    "teacher_b_s2_sequence_kd_reproduction_policy.json"
)
write_json(SEQUENCE_POLICY_PATH, SEQUENCE_KD_POLICY)

write_json(
    AUDIT_DIR / "phase4g_cell04_sequence_policy.json",
    {
        "audit_version": "phase4g_cell04_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "alpha": 0.5,
        "temperature_applicable": False,
        "direct_teacher_logits_used": False,
        "teacher_generation": "Beam-3 deterministic",
        "pseudo_caption_split": "Train+Restval only",
        "validation_used_for_selection_only": True,
        "feature_kd_deferred": True,
        "training_started": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 04G SEQUENCE-KD POLICY")
print("=" * 72)
print("Status                    : LOCKED")
print("Teacher generation        : Beam-3 / deterministic")
print("Pseudo-caption split      : Train+Restval only")
print("Human / teacher CE weight : 0.5 / 0.5")
print("Temperature               : NOT APPLICABLE")
print("Direct logits used        : False")
print("Validation                : selection/evaluation only")
print("Feature KD                : deferred to Phase 5")
print("Training started          : False")
print("Test Set status           : SEALED")
print("Next                      : package and authorize Phase 04H")


PHASE 04G SEQUENCE-KD POLICY
Status                    : LOCKED
Teacher generation        : Beam-3 / deterministic
Pseudo-caption split      : Train+Restval only
Human / teacher CE weight : 0.5 / 0.5
Temperature               : NOT APPLICABLE
Direct logits used        : False
Validation                : selection/evaluation only
Feature KD                : deferred to Phase 5
Training started          : False
Test Set status           : SEALED
Next                      : package and authorize Phase 04H


## Cell 5 — Formal package và Phase 04H authorization

Tạo manifest, complete marker và authorization cho GPU preflight.


In [5]:
CELL_STARTED_AT = time.perf_counter()
NEXT_AUTHORIZATION = OUTPUT_ROOT / (
    "PHASE4_TEACHER_B_S2_SEQUENCE_KD_PREFLIGHT_AUTHORIZED.json"
)
write_json(
    NEXT_AUTHORIZATION,
    {
        "marker_version": (
            "phase4_teacher_b_s2_sequence_kd_"
            "preflight_authorized_v1"
        ),
        "created_at_utc": now_utc(),
        "status": "authorized",
        "experiment_id": EXPERIMENT_ID,
        "teacher_id": "B",
        "student_id": "S2",
        "development_seed": 42,
        "selected_method": "sequence_kd",
        "direct_logit_kd_allowed": False,
        "feature_kd_allowed_in_phase4": False,
        "policy": {
            "relative_path": SEQUENCE_POLICY_PATH.relative_to(
                OUTPUT_ROOT
            ).as_posix(),
            "sha256": sha256_file(SEQUENCE_POLICY_PATH),
        },
        "next_phase": (
            "Phase 04H Teacher B -> S2 Sequence-KD "
            "GPU preflight"
        ),
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)

MANIFEST_CSV = MANIFEST_DIR / "phase4g_manifest.csv"
MANIFEST_JSON = MANIFEST_DIR / "phase4g_manifest.json"
SHA256SUMS = MANIFEST_DIR / "phase4g_SHA256SUMS.txt"
FINAL_MARKER = OUTPUT_ROOT / (
    "PHASE4_TEACHER_B_S2_KD_COMPATIBILITY_ENTRY_COMPLETE.json"
)
artifact_paths = sorted(
    [
        path for path in OUTPUT_ROOT.rglob("*")
        if path.is_file()
        and path not in {
            MANIFEST_CSV, MANIFEST_JSON, SHA256SUMS, FINAL_MARKER
        }
        and not path.name.endswith(".tmp")
    ],
    key=lambda path: path.relative_to(OUTPUT_ROOT).as_posix(),
)
manifest_rows = [
    {
        "relative_path": path.relative_to(
            OUTPUT_ROOT
        ).as_posix(),
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    }
    for path in artifact_paths
]
with MANIFEST_CSV.open(
    "w", encoding="utf-8", newline=""
) as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=["relative_path", "size_bytes", "sha256"],
    )
    writer.writeheader()
    writer.writerows(manifest_rows)
write_json(
    MANIFEST_JSON,
    {
        "schema_version": "phase4g_manifest_v1",
        "created_at_utc": now_utc(),
        "artifact_count": len(manifest_rows),
        "artifacts": manifest_rows,
    },
)
SHA256SUMS.write_text(
    "".join(
        f'{row["sha256"]}  {row["relative_path"]}\n'
        for row in manifest_rows
    ),
    encoding="utf-8",
)
for row in manifest_rows:
    artifact = resolve_inside(
        OUTPUT_ROOT, row["relative_path"]
    )
    assert artifact.stat().st_size == int(row["size_bytes"])
    assert sha256_file(artifact) == row["sha256"]

write_json(
    FINAL_MARKER,
    {
        "marker_version": (
            "phase4_teacher_b_s2_kd_"
            "compatibility_entry_complete_v1"
        ),
        "created_at_utc": now_utc(),
        "status": "complete",
        "experiment_id": EXPERIMENT_ID,
        "teacher": "Teacher B — BLIP Base",
        "student": "S2 — MobileViT-S",
        "development_seed": 42,
        "compatibility_decision": {
            "relative_path": COMPATIBILITY_PATH.relative_to(
                OUTPUT_ROOT
            ).as_posix(),
            "sha256": sha256_file(COMPATIBILITY_PATH),
        },
        "selected_method": "sequence_kd",
        "direct_logit_kd": "prohibited",
        "feature_kd": "deferred_to_phase5",
        "phase04h_authorized": True,
        "next_authorization": {
            "relative_path": NEXT_AUTHORIZATION.relative_to(
                OUTPUT_ROOT
            ).as_posix(),
            "sha256": sha256_file(NEXT_AUTHORIZATION),
        },
        "manifest": {
            "csv": {
                "relative_path": MANIFEST_CSV.relative_to(
                    OUTPUT_ROOT
                ).as_posix(),
                "sha256": sha256_file(MANIFEST_CSV),
            },
            "json": {
                "relative_path": MANIFEST_JSON.relative_to(
                    OUTPUT_ROOT
                ).as_posix(),
                "sha256": sha256_file(MANIFEST_JSON),
            },
            "sha256sums": {
                "relative_path": SHA256SUMS.relative_to(
                    OUTPUT_ROOT
                ).as_posix(),
                "sha256": sha256_file(SHA256SUMS),
            },
        },
        "manifest_rows": len(manifest_rows),
        "manifest_verification": "all_passed",
        "training_started": False,
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)

print("=" * 72)
print("PHASE 04G CELL 5 STATUS")
print("=" * 72)
print("Status                    : COMPLETE")
print("Teacher / Student         : B / S2")
print("Direct Logit-KD           : PROHIBITED")
print("Selected reproduction KD  : SEQUENCE KD")
print("Feature KD                : DEFERRED TO PHASE 5")
print("Manifest rows             :", len(manifest_rows))
print("Manifest verification     : ALL PASSED")
print("Training started          : False")
print("Test Set status           : SEALED")
print("Phase 04H GPU preflight   : AUTHORIZED")
print("Marker                    :", FINAL_MARKER)


PHASE 04G CELL 5 STATUS
Status                    : COMPLETE
Teacher / Student         : B / S2
Direct Logit-KD           : PROHIBITED
Selected reproduction KD  : SEQUENCE KD
Feature KD                : DEFERRED TO PHASE 5
Manifest rows             : 7
Manifest verification     : ALL PASSED
Training started          : False
Test Set status           : SEALED
Phase 04H GPU preflight   : AUTHORIZED
Marker                    : /kaggle/working//phase4_teacher_b_s2_kd_compatibility_entry/PHASE4_TEACHER_B_S2_KD_COMPATIBILITY_ENTRY_COMPLETE.json
